# Chapter 9 — Parallel histogram

Runs every program of `ec2/histogram` on a Colab GPU. First choose **Runtime → Change runtime type → GPU (T4 is enough)**, then **Runtime → Run all**.

The theory is in `ec2/histogram/README.md`. The same `.cu` files also run on a laptop without a GPU: `clang++ -std=c++17 -O2 -pthread -x c++ -I../common FILE.cu -o FILE && ./FILE`.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

The compatibility header: plain CUDA under `nvcc`, a CPU emulation under `g++`/`clang++`.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

inline const char* cudaGetErrorString(cudaError_t) {
  return "no error";
}

inline cudaError_t cudaGetLastError() {
  return cudaSuccess;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_station_hour.cu`

In [ ]:
%%writefile bikeshare_station_hour.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

constexpr unsigned HOURS = 24;
constexpr unsigned STATION_COLUMNS = 8;
constexpr unsigned STATION_ROWS = 6;
constexpr unsigned NUM_STATIONS = STATION_COLUMNS * STATION_ROWS;
constexpr unsigned BINS_B = NUM_STATIONS * HOURS;
constexpr unsigned MS_PER_HOUR = 60 * 60 * 1000;
constexpr unsigned THREADS_PER_BLOCK = 256;
constexpr unsigned CFACTOR = 64;

#ifndef CPU_RENTALS
#define CPU_RENTALS (1u << 19)
#endif
const unsigned NUM_SYNTHETIC_RENTALS = BY_BACKEND(1u << 24, CPU_RENTALS);
const int TIMED_RUNS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

__host__ __device__ inline unsigned hourOf(unsigned tMs) {
  return (tMs / MS_PER_HOUR) % HOURS;
}

__host__ __device__ inline unsigned stationHourBin(unsigned station, unsigned tMs) {
  return station * HOURS + hourOf(tMs);
}

__global__ void perHourInterleaved(const unsigned* startMs, unsigned n, unsigned* histo) {
  __shared__ unsigned histo_s[HOURS];
  for (unsigned b = threadIdx.x; b < HOURS; b += blockDim.x) {
    histo_s[b] = 0u;
  }
  __syncthreads();

  unsigned accumulator = 0;
  int prevHour = -1;
  for (unsigned i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
    int hour = static_cast<int>(hourOf(startMs[i]));
    if (hour == prevHour) {
      ++accumulator;
    } else {
      if (accumulator > 0) {
        atomicAdd(&histo_s[prevHour], accumulator);
      }
      accumulator = 1;
      prevHour = hour;
    }
  }
  if (accumulator > 0) {
    atomicAdd(&histo_s[prevHour], accumulator);
  }
  __syncthreads();

  for (unsigned b = threadIdx.x; b < HOURS; b += blockDim.x) {
    if (histo_s[b] > 0) {
      atomicAdd(&histo[b], histo_s[b]);
    }
  }
}

__global__ void perHourContiguous(const unsigned* startMs, unsigned n, unsigned* histo) {
  __shared__ unsigned histo_s[HOURS];
  for (unsigned b = threadIdx.x; b < HOURS; b += blockDim.x) {
    histo_s[b] = 0u;
  }
  __syncthreads();

  unsigned accumulator = 0;
  int prevHour = -1;
  unsigned begin = (blockIdx.x * blockDim.x + threadIdx.x) * CFACTOR;
  unsigned end = begin + CFACTOR < n ? begin + CFACTOR : n;
  for (unsigned i = begin; i < end; ++i) {
    int hour = static_cast<int>(hourOf(startMs[i]));
    if (hour == prevHour) {
      ++accumulator;
    } else {
      if (accumulator > 0) {
        atomicAdd(&histo_s[prevHour], accumulator);
      }
      accumulator = 1;
      prevHour = hour;
    }
  }
  if (accumulator > 0) {
    atomicAdd(&histo_s[prevHour], accumulator);
  }
  __syncthreads();

  for (unsigned b = threadIdx.x; b < HOURS; b += blockDim.x) {
    if (histo_s[b] > 0) {
      atomicAdd(&histo[b], histo_s[b]);
    }
  }
}

__global__ void perStationHour(const unsigned* startMs,
                               const unsigned* startStation,
                               unsigned n,
                               unsigned* histo) {
  __shared__ unsigned histo_s[BINS_B];
  for (unsigned b = threadIdx.x; b < BINS_B; b += blockDim.x) {
    histo_s[b] = 0u;
  }
  __syncthreads();

  for (unsigned i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
    atomicAdd(&histo_s[stationHourBin(startStation[i], startMs[i])], 1u);
  }
  __syncthreads();

  for (unsigned b = threadIdx.x; b < BINS_B; b += blockDim.x) {
    if (histo_s[b] > 0) {
      atomicAdd(&histo[b], histo_s[b]);
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  double uniformOpenInterval() {
    const uint64_t mantissa = next() >> (64 - DOUBLE_MANTISSA_BITS);
    return (static_cast<double>(mantissa) + 0.5) * (1.0 / TWO_TO_THE_MANTISSA_BITS);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;
  static constexpr int DOUBLE_MANTISSA_BITS = 53;
  static constexpr double TWO_TO_THE_MANTISSA_BITS =
      static_cast<double>(1ull << DOUBLE_MANTISSA_BITS);

  uint64_t state_;
};

struct Position {
  int x;
  int y;
};

struct Rentals {
  std::vector<unsigned> startMs;
  std::vector<unsigned> startStation;
};

constexpr double RELATIVE_RENTAL_RATE_BY_HOUR[HOURS] = {
    0.10, 0.05, 0.03, 0.03, 0.05, 0.20, 0.80, 2.20, 3.00, 1.60, 0.90, 0.90,
    1.20, 1.10, 0.90, 1.00, 1.50, 2.40, 3.10, 2.00, 1.10, 0.70, 0.40, 0.20,
};
constexpr int CITY_SIDE_M = 6000;
constexpr int STATION_SPACING_X_M = CITY_SIDE_M / STATION_COLUMNS;
constexpr int STATION_SPACING_Y_M = CITY_SIDE_M / STATION_ROWS;
constexpr double METRES_PER_KM = 1000;
constexpr double BASE_POPULARITY = 1.0;
constexpr double CENTRE_BONUS = 6.0;
constexpr unsigned SYNTHETIC_DAYS = 7;
constexpr uint64_t SYNTHETIC_SEED = 7;

static Position stationPosition(unsigned station) {
  const int column = static_cast<int>(station % STATION_COLUMNS);
  const int row = static_cast<int>(station / STATION_COLUMNS);
  return {column * STATION_SPACING_X_M + STATION_SPACING_X_M / 2,
          row * STATION_SPACING_Y_M + STATION_SPACING_Y_M / 2};
}

static double popularity(unsigned station) {
  const Position position = stationPosition(station);
  const double dxKm = (position.x - CITY_SIDE_M / 2) / METRES_PER_KM;
  const double dyKm = (position.y - CITY_SIDE_M / 2) / METRES_PER_KM;
  const double distanceToCentreKm = std::sqrt(dxKm * dxKm + dyKm * dyKm);
  return BASE_POPULARITY + CENTRE_BONUS / (1.0 + distanceToCentreKm);
}

static std::vector<double> cumulativePopularity() {
  std::vector<double> cumulative(NUM_STATIONS);
  double sum = 0;
  for (unsigned s = 0; s < NUM_STATIONS; ++s) {
    sum += popularity(s);
    cumulative[s] = sum;
  }
  return cumulative;
}

static unsigned pickStationByPopularity(SplitMix64& rng, const std::vector<double>& cumulative) {
  const double draw = rng.uniformOpenInterval() * cumulative.back();
  for (unsigned s = 0; s < NUM_STATIONS; ++s) {
    if (draw < cumulative[s]) {
      return s;
    }
  }
  return NUM_STATIONS - 1;
}

static double nextRentalMs(double nowMs, double meanGapMs, SplitMix64& rng) {
  const double exponentialGap = -std::log(rng.uniformOpenInterval()) * meanGapMs;
  const double rateNow = RELATIVE_RENTAL_RATE_BY_HOUR[hourOf(static_cast<unsigned>(nowMs))];
  return nowMs + exponentialGap / rateNow;
}

static Rentals syntheticRentals(unsigned count, uint64_t seed) {
  SplitMix64 rng(seed);
  const std::vector<double> cumulative = cumulativePopularity();
  Rentals rentals;
  rentals.startMs.resize(count);
  rentals.startStation.resize(count);

  const double meanGapMs = static_cast<double>(SYNTHETIC_DAYS) * HOURS * MS_PER_HOUR / count;
  double nowMs = 0;
  for (unsigned i = 0; i < count; ++i) {
    nowMs = nextRentalMs(nowMs, meanGapMs, rng);
    rentals.startMs[i] = static_cast<unsigned>(nowMs);
    rentals.startStation[i] = pickStationByPopularity(rng, cumulative);
  }
  return rentals;
}

using Histogram = std::vector<unsigned>;

struct References {
  Histogram perHour;
  Histogram stationHour;
  double cpuMs;
};

static References sequentialReferences(const Rentals& rentals) {
  References references;
  references.perHour.assign(HOURS, 0);
  references.stationHour.assign(BINS_B, 0);
  Timer timer;
  for (size_t i = 0; i < rentals.startMs.size(); ++i) {
    references.perHour[hourOf(rentals.startMs[i])]++;
    references.stationHour[stationHourBin(rentals.startStation[i], rentals.startMs[i])]++;
  }
  references.cpuMs = timer.ms();
  return references;
}

enum class Partition { Interleaved, Contiguous };

static unsigned long long countAggregatedAtomics(const std::vector<unsigned>& startMs,
                                                 unsigned threadCount,
                                                 Partition partition) {
  const unsigned count = static_cast<unsigned>(startMs.size());
  const unsigned distanceToPreviousOfSameThread =
      partition == Partition::Contiguous ? 1 : threadCount;
  unsigned long long runs = 0;
  for (unsigned i = 0; i < count; ++i) {
    const bool startsThreadWork =
        partition == Partition::Contiguous ? i % CFACTOR == 0 : i < threadCount;
    if (startsThreadWork ||
        hourOf(startMs[i]) != hourOf(startMs[i - distanceToPreviousOfSameThread])) {
      ++runs;
    }
  }
  return runs;
}

struct DeviceBuffers {
  unsigned* startMs;
  unsigned* startStation;
  unsigned* perHour;
  unsigned* stationHour;
  unsigned count;
  unsigned blocks;
};

static unsigned atLeastOne(unsigned count) {
  return count > 0 ? count : 1;
}

static DeviceBuffers copyRentalsToDevice(const Rentals& rentals) {
  DeviceBuffers device;
  device.count = static_cast<unsigned>(rentals.startMs.size());
  device.blocks = device.count > 0 ? cdiv(device.count, THREADS_PER_BLOCK * CFACTOR) : 1;

  const size_t inputBytes = atLeastOne(device.count) * sizeof(unsigned);
  CHECK(cudaMalloc(&device.startMs, inputBytes));
  CHECK(cudaMalloc(&device.startStation, inputBytes));
  CHECK(cudaMalloc(&device.perHour, HOURS * sizeof(unsigned)));
  CHECK(cudaMalloc(&device.stationHour, BINS_B * sizeof(unsigned)));

  if (device.count > 0) {
    const size_t bytes = device.count * sizeof(unsigned);
    CHECK(cudaMemcpy(device.startMs, rentals.startMs.data(), bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(
        device.startStation, rentals.startStation.data(), bytes, cudaMemcpyHostToDevice));
  }
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.startMs));
  CHECK(cudaFree(device.startStation));
  CHECK(cudaFree(device.perHour));
  CHECK(cudaFree(device.stationHour));
}

template <class ClearAndLaunch>
static double timeLaunches(ClearAndLaunch clearAndLaunch) {
  if (WARM_UP_BEFORE_TIMING) {
    clearAndLaunch();
  }
  Timer timer;
  for (int run = 0; run < TIMED_RUNS; ++run) {
    clearAndLaunch();
  }
  const double ms = timer.ms() / TIMED_RUNS;
  CHECK(cudaGetLastError());
  return ms;
}

static Histogram copyToHost(const unsigned* d_histo, unsigned bins) {
  Histogram histo(bins);
  CHECK(cudaMemcpy(histo.data(), d_histo, bins * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return histo;
}

static void printRow(
    const char* name, unsigned blocks, double ms, unsigned long long sharedAtomics, bool correct) {
  std::printf(
      "%-34s %7u %10.3f %15llu  %s\n", name, blocks, ms, sharedAtomics, correct ? "PASS" : "FAIL");
}

using PerHourKernel = void (*)(const unsigned*, unsigned, unsigned*);

static bool runPerHourKernel(const char* name,
                             PerHourKernel kernel,
                             Partition partition,
                             const DeviceBuffers& device,
                             const Rentals& rentals,
                             const Histogram& reference,
                             Histogram& perHour) {
  const unsigned* startMs = device.startMs;
  const double ms = timeLaunches([&] {
    CHECK(cudaMemset(device.perHour, 0, HOURS * sizeof(unsigned)));
    launch(kernel,
           dim3(device.blocks),
           dim3(THREADS_PER_BLOCK),
           startMs,
           device.count,
           device.perHour);
  });

  perHour = copyToHost(device.perHour, HOURS);
  const bool correct = perHour == reference;
  const unsigned threadCount = device.blocks * THREADS_PER_BLOCK;
  printRow(name,
           device.blocks,
           ms,
           countAggregatedAtomics(rentals.startMs, threadCount, partition),
           correct);
  return correct;
}

static bool runPerStationHourKernel(const DeviceBuffers& device,
                                    const Histogram& reference,
                                    Histogram& stationHourBins) {
  const unsigned* startMs = device.startMs;
  const unsigned* startStation = device.startStation;
  const double ms = timeLaunches([&] {
    CHECK(cudaMemset(device.stationHour, 0, BINS_B * sizeof(unsigned)));
    launch(perStationHour,
           dim3(device.blocks),
           dim3(THREADS_PER_BLOCK),
           startMs,
           startStation,
           device.count,
           device.stationHour);
  });

  stationHourBins = copyToHost(device.stationHour, BINS_B);
  const bool correct = stationHourBins == reference;
  const unsigned long long oneAtomicPerRental = device.count;
  printRow("B station x hour, shared priv.", device.blocks, ms, oneAtomicPerRental, correct);
  return correct;
}

struct GpuHistograms {
  Histogram perHour;
  Histogram stationHour;
};

static bool runKernels(const Rentals& rentals, const References& references, GpuHistograms& gpu) {
  const DeviceBuffers device = copyRentalsToDevice(rentals);

  std::printf("\n%-34s %7s %10s %15s  %s\n", "kernel", "blocks", "ms", "shared atomics", "check");
  std::printf("%-34s %7s %10.3f %15s  %s\n",
              "CPU sequential (A and B)",
              "-",
              references.cpuMs,
              "-",
              "reference");

  const bool interleavedCorrect = runPerHourKernel("A per hour, interleaved + aggr.",
                                                   perHourInterleaved,
                                                   Partition::Interleaved,
                                                   device,
                                                   rentals,
                                                   references.perHour,
                                                   gpu.perHour);
  const bool contiguousCorrect = runPerHourKernel("A per hour, contiguous + aggr.",
                                                  perHourContiguous,
                                                  Partition::Contiguous,
                                                  device,
                                                  rentals,
                                                  references.perHour,
                                                  gpu.perHour);
  const bool stationHourCorrect =
      runPerStationHourKernel(device, references.stationHour, gpu.stationHour);

  freeDeviceBuffers(device);
  return interleavedCorrect && contiguousCorrect && stationHourCorrect;
}

constexpr int BAR_WIDTH = 50;

static void printBar(unsigned count, unsigned peak) {
  const int width =
      peak ? static_cast<int>(BAR_WIDTH * static_cast<double>(count) / peak + 0.5) : 0;
  for (int j = 0; j < width; ++j) {
    std::putchar('#');
  }
  std::putchar('\n');
}

static unsigned largest(const Histogram& histo) {
  unsigned peak = 0;
  for (unsigned count : histo) {
    if (count > peak) {
      peak = count;
    }
  }
  return peak;
}

static unsigned binCount(const Histogram& stationHour, unsigned station, unsigned hour) {
  return stationHour[station * HOURS + hour];
}

static Histogram rowSums(const Histogram& stationHour) {
  Histogram perStation(NUM_STATIONS, 0);
  for (unsigned s = 0; s < NUM_STATIONS; ++s) {
    for (unsigned h = 0; h < HOURS; ++h) {
      perStation[s] += binCount(stationHour, s, h);
    }
  }
  return perStation;
}

static bool columnSumsEqual(const Histogram& stationHour, const Histogram& perHour) {
  for (unsigned h = 0; h < HOURS; ++h) {
    unsigned columnSum = 0;
    for (unsigned s = 0; s < NUM_STATIONS; ++s) {
      columnSum += binCount(stationHour, s, h);
    }
    if (columnSum != perHour[h]) {
      return false;
    }
  }
  return true;
}

struct StationHour {
  unsigned station;
  unsigned hour;
};

static StationHour busiestStationHour(const Histogram& stationHour) {
  StationHour busiest = {0, 0};
  for (unsigned s = 0; s < NUM_STATIONS; ++s) {
    for (unsigned h = 0; h < HOURS; ++h) {
      const unsigned busiestCount = binCount(stationHour, busiest.station, busiest.hour);
      if (binCount(stationHour, s, h) > busiestCount) {
        busiest = {s, h};
      }
    }
  }
  return busiest;
}

static void printRentalsPerHour(const Histogram& perHour) {
  const unsigned peak = largest(perHour);
  std::printf("\nhour   rentals\n");
  for (unsigned h = 0; h < HOURS; ++h) {
    std::printf("%02u  %9u  ", h, perHour[h]);
    printBar(perHour[h], peak);
  }
}

static void printRentalsPerStationAsMap(const Histogram& perStation) {
  std::printf("\nrentals per station, laid out as on the map (row 0 is the south edge)\n");
  for (int row = STATION_ROWS - 1; row >= 0; --row) {
    std::printf("row %d ", row);
    for (unsigned column = 0; column < STATION_COLUMNS; ++column) {
      std::printf(" %7u", perStation[row * STATION_COLUMNS + column]);
    }
    std::printf("\n");
  }
}

static bool printReport(const Rentals& rentals, const GpuHistograms& gpu) {
  printRentalsPerHour(gpu.perHour);
  printRentalsPerStationAsMap(rowSums(gpu.stationHour));

  if (!rentals.startMs.empty()) {
    const StationHour busiest = busiestStationHour(gpu.stationHour);
    std::printf("\nbusiest station-hour: s%02u at %02u:00 with %u rentals\n",
                busiest.station,
                busiest.hour,
                binCount(gpu.stationHour, busiest.station, busiest.hour));
  }

  const bool consistent = columnSumsEqual(gpu.stationHour, gpu.perHour);
  std::printf("columns of B equal A: %s\n", consistent ? "yes" : "NO");
  return consistent;
}

int main() {
  std::printf("Bike rentals per hour and per station (chapter 9 use case), backend: %s\n",
              CUDA_BACKEND);
  const Rentals rentals = syntheticRentals(NUM_SYNTHETIC_RENTALS, SYNTHETIC_SEED);
  std::printf("synthetic week, %zu rentals at %u stations (seed %llu)\n",
              rentals.startMs.size(),
              NUM_STATIONS,
              static_cast<unsigned long long>(SYNTHETIC_SEED));
  const References references = sequentialReferences(rentals);

  GpuHistograms gpu;
  const bool kernelsCorrect = runKernels(rentals, references, gpu);
  const bool consistent = printReport(rentals, gpu);

  const bool allPass = kernelsCorrect && consistent;
  std::printf("%s\n", allPass ? "ALL PASS" : "SOME TESTS FAILED");
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_station_hour.cu -o bikeshare_station_hour && ./bikeshare_station_hour

## `bikeshare_trip_durations.cu`

In [ ]:
%%writefile bikeshare_trip_durations.cu
#include "cuda_compat.h"

#include <cstdint>
#include <cstdio>
#include <vector>

constexpr unsigned MS_PER_SECOND = 1000;
constexpr unsigned MS_PER_MINUTE = 60 * MS_PER_SECOND;
constexpr unsigned MINUTES_WITH_OWN_BUCKET = 60;
constexpr unsigned NUM_BUCKETS = MINUTES_WITH_OWN_BUCKET + 1;
constexpr unsigned OVERFLOW_BUCKET = NUM_BUCKETS - 1;
constexpr unsigned MS_PER_BUCKET = MS_PER_MINUTE;
constexpr unsigned THREADS_PER_BLOCK = 256;
constexpr unsigned CFACTOR = 32;

#ifndef CPU_TRIPS
#define CPU_TRIPS (1u << 18)
#endif
const unsigned NUM_SYNTHETIC_TRIPS = BY_BACKEND(1u << 24, CPU_TRIPS);
const int TIMED_RUNS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

__host__ __device__ inline unsigned bucketOf(unsigned ms) {
  unsigned b = ms / MS_PER_BUCKET;
  return b < OVERFLOW_BUCKET ? b : OVERFLOW_BUCKET;
}

__global__ void tripHistoBasic(const unsigned* tripMs, unsigned n, unsigned* histo) {
  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    atomicAdd(&histo[bucketOf(tripMs[i])], 1u);
  }
}

__global__ void tripHistoPrivate(const unsigned* tripMs, unsigned n, unsigned* histo) {
  __shared__ unsigned histo_s[NUM_BUCKETS];
  for (unsigned b = threadIdx.x; b < NUM_BUCKETS; b += blockDim.x) {
    histo_s[b] = 0u;
  }
  __syncthreads();

  for (unsigned i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
    atomicAdd(&histo_s[bucketOf(tripMs[i])], 1u);
  }
  __syncthreads();

  for (unsigned b = threadIdx.x; b < NUM_BUCKETS; b += blockDim.x) {
    unsigned v = histo_s[b];
    if (v > 0) {
      atomicAdd(&histo[b], v);
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  bool oneIn(unsigned n) {
    return next() % n == 0;
  }

  bool percentChance(unsigned percent) {
    return next() % 100 < percent;
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

struct Range {
  unsigned lowest;
  unsigned highest;
};

static unsigned randomIn(SplitMix64& rng, Range range) {
  return range.lowest + static_cast<unsigned>(rng.next() % (range.highest - range.lowest + 1));
}

constexpr Range UNLOCK_AND_ADJUST_SEAT = {20 * MS_PER_SECOND, 90 * MS_PER_SECOND};
constexpr Range RIDE_DISTANCE_M = {600, 4500};
constexpr unsigned E_BIKE_PERCENT = 40;
constexpr unsigned PEDAL_BIKE_METRES_PER_MINUTE = 250;
constexpr unsigned E_BIKE_METRES_PER_MINUTE = 330;
constexpr Range TRAFFIC_LIGHTS_AND_HILLS = {0, 4 * MS_PER_MINUTE};
constexpr unsigned ONE_FULL_END_STATION_EVERY = 6;
constexpr Range SEARCH_FOR_A_FREE_DOCK = {1 * MS_PER_MINUTE, 6 * MS_PER_MINUTE};
constexpr unsigned LEISURE_RIDE_PERCENT = 3;
constexpr Range LEISURE_DETOUR = {10 * MS_PER_MINUTE, 40 * MS_PER_MINUTE};
constexpr uint64_t SYNTHETIC_SEED = 42;

static unsigned ridingMs(unsigned distanceM, unsigned metresPerMinute) {
  return distanceM * MS_PER_MINUTE / metresPerMinute;
}

static unsigned syntheticTrip(SplitMix64& rng) {
  const unsigned unlock = randomIn(rng, UNLOCK_AND_ADJUST_SEAT);
  const unsigned distanceM = randomIn(rng, RIDE_DISTANCE_M);
  const bool eBike = rng.percentChance(E_BIKE_PERCENT);
  const unsigned speed = eBike ? E_BIKE_METRES_PER_MINUTE : PEDAL_BIKE_METRES_PER_MINUTE;

  unsigned ride = ridingMs(distanceM, speed);
  ride += randomIn(rng, TRAFFIC_LIGHTS_AND_HILLS);

  unsigned dockSearch = 0;
  if (rng.oneIn(ONE_FULL_END_STATION_EVERY)) {
    dockSearch = randomIn(rng, SEARCH_FOR_A_FREE_DOCK);
  }

  unsigned detour = 0;
  if (rng.percentChance(LEISURE_RIDE_PERCENT)) {
    detour = randomIn(rng, LEISURE_DETOUR);
  }

  return unlock + ride + dockSearch + detour;
}

static std::vector<unsigned> syntheticTrips(unsigned count, uint64_t seed) {
  SplitMix64 rng(seed);
  std::vector<unsigned> tripMs(count);
  for (unsigned& ms : tripMs) {
    ms = syntheticTrip(rng);
  }
  return tripMs;
}

using Histogram = std::vector<unsigned>;

static Histogram sequentialHistogram(const std::vector<unsigned>& tripMs, double& elapsedMs) {
  Histogram histo(NUM_BUCKETS, 0);
  Timer timer;
  for (unsigned ms : tripMs) {
    histo[bucketOf(ms)]++;
  }
  elapsedMs = timer.ms();
  return histo;
}

static unsigned atLeastOne(unsigned count) {
  return count > 0 ? count : 1;
}

using TripKernel = void (*)(const unsigned*, unsigned, unsigned*);

struct KernelVariant {
  const char* name;
  TripKernel kernel;
  unsigned blocks;
};

static unsigned blocksFor(unsigned count, unsigned tripsPerThread) {
  if (count == 0) {
    return 1;
  }
  return cdiv(count, THREADS_PER_BLOCK * tripsPerThread);
}

static std::vector<KernelVariant> kernelVariants(unsigned count) {
  const unsigned oneTripPerThread = 1;
  return {
      {"basic, global atomics", tripHistoBasic, blocksFor(count, oneTripPerThread)},
      {"shared private + coarsened", tripHistoPrivate, blocksFor(count, CFACTOR)},
  };
}

struct DeviceBuffers {
  unsigned* tripMs;
  unsigned* histo;
  unsigned count;
};

static DeviceBuffers copyTripsToDevice(const std::vector<unsigned>& tripMs) {
  DeviceBuffers device;
  device.count = static_cast<unsigned>(tripMs.size());
  CHECK(cudaMalloc(&device.tripMs, atLeastOne(device.count) * sizeof(unsigned)));
  CHECK(cudaMalloc(&device.histo, NUM_BUCKETS * sizeof(unsigned)));
  if (device.count > 0) {
    CHECK(cudaMemcpy(
        device.tripMs, tripMs.data(), device.count * sizeof(unsigned), cudaMemcpyHostToDevice));
  }
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.tripMs));
  CHECK(cudaFree(device.histo));
}

static double timeKernel(const KernelVariant& variant, const DeviceBuffers& device) {
  const unsigned* tripMs = device.tripMs;
  auto clearAndLaunch = [&] {
    CHECK(cudaMemset(device.histo, 0, NUM_BUCKETS * sizeof(unsigned)));
    launch(variant.kernel,
           dim3(variant.blocks),
           dim3(THREADS_PER_BLOCK),
           tripMs,
           device.count,
           device.histo);
  };

  if (WARM_UP_BEFORE_TIMING) {
    clearAndLaunch();
  }
  Timer timer;
  for (int run = 0; run < TIMED_RUNS; ++run) {
    clearAndLaunch();
  }
  const double ms = timer.ms() / TIMED_RUNS;
  CHECK(cudaGetLastError());
  return ms;
}

static Histogram copyHistogramToHost(const DeviceBuffers& device) {
  Histogram histo(NUM_BUCKETS);
  CHECK(cudaMemcpy(
      histo.data(), device.histo, NUM_BUCKETS * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return histo;
}

static bool runKernels(const std::vector<unsigned>& tripMs,
                       const Histogram& reference,
                       double cpuMs,
                       Histogram& gpuHistogram) {
  const DeviceBuffers device = copyTripsToDevice(tripMs);

  std::printf("\n%-28s %8s %10s  %s\n", "kernel", "blocks", "ms", "check");
  std::printf("%-28s %8s %10.3f  %s\n", "CPU sequential", "-", cpuMs, "reference");
  bool allCorrect = true;
  for (const KernelVariant& variant : kernelVariants(device.count)) {
    const double ms = timeKernel(variant, device);
    gpuHistogram = copyHistogramToHost(device);
    const bool correct = gpuHistogram == reference;
    allCorrect = allCorrect && correct;
    std::printf(
        "%-28s %8u %10.3f  %s\n", variant.name, variant.blocks, ms, correct ? "PASS" : "FAIL");
  }

  freeDeviceBuffers(device);
  return allCorrect;
}

constexpr int BAR_WIDTH = 60;
constexpr unsigned INCLUDED_MINUTES = 45;
constexpr double MEDIAN_FRACTION = 0.50;
constexpr double P90_FRACTION = 0.90;
constexpr double P99_FRACTION = 0.99;

static unsigned long long totalCount(const Histogram& histo) {
  unsigned long long total = 0;
  for (unsigned count : histo) {
    total += count;
  }
  return total;
}

static unsigned long long countBelowMinute(const Histogram& histo, unsigned minute) {
  unsigned long long count = 0;
  for (unsigned b = 0; b < minute; ++b) {
    count += histo[b];
  }
  return count;
}

static unsigned fullestBucketCount(const Histogram& histo) {
  unsigned peak = 0;
  for (unsigned count : histo) {
    if (count > peak) {
      peak = count;
    }
  }
  return peak;
}

static unsigned percentileMinute(const Histogram& histo,
                                 unsigned long long total,
                                 double fraction) {
  const unsigned long long needed = static_cast<unsigned long long>(fraction * total + 0.5);
  unsigned long long runningSum = 0;
  for (unsigned b = 0; b < NUM_BUCKETS; ++b) {
    runningSum += histo[b];
    if (runningSum >= needed) {
      return b;
    }
  }
  return OVERFLOW_BUCKET;
}

static void printBar(unsigned count, unsigned peak) {
  const int width = static_cast<int>(BAR_WIDTH * static_cast<double>(count) / peak + 0.5);
  for (int j = 0; j < width; ++j) {
    std::putchar('#');
  }
  std::putchar('\n');
}

static void printMinuteHistogram(const Histogram& histo) {
  const unsigned peak = fullestBucketCount(histo);
  std::printf("\nminutes     trips\n");
  for (unsigned b = 0; b < NUM_BUCKETS; ++b) {
    if (histo[b] == 0) {
      continue;
    }
    const char* prefix = b == OVERFLOW_BUCKET ? ">=" : "  ";
    std::printf("%s%2u  %9u  ", prefix, b, histo[b]);
    printBar(histo[b], peak);
  }
}

static void printPercentiles(const Histogram& histo) {
  const unsigned long long total = totalCount(histo);
  if (total == 0) {
    return;
  }
  const unsigned long long withinIncludedTime = countBelowMinute(histo, INCLUDED_MINUTES);
  std::printf("\nmedian %u min, p90 %u min, p99 %u min\n",
              percentileMinute(histo, total, MEDIAN_FRACTION),
              percentileMinute(histo, total, P90_FRACTION),
              percentileMinute(histo, total, P99_FRACTION));
  std::printf("within the included %u min: %.1f%%, late (%u min or more): %llu trips\n",
              INCLUDED_MINUTES,
              100.0 * withinIncludedTime / total,
              INCLUDED_MINUTES,
              total - withinIncludedTime);
}

int main() {
  std::printf("Bike-share trip durations (chapter 9 use case), backend: %s\n", CUDA_BACKEND);
  const std::vector<unsigned> tripMs = syntheticTrips(NUM_SYNTHETIC_TRIPS, SYNTHETIC_SEED);
  std::printf("%zu synthetic trips (seed %llu)\n",
              tripMs.size(),
              static_cast<unsigned long long>(SYNTHETIC_SEED));

  double cpuMs;
  const Histogram reference = sequentialHistogram(tripMs, cpuMs);

  Histogram gpuHistogram;
  const bool allCorrect = runKernels(tripMs, reference, cpuMs, gpuHistogram);

  printMinuteHistogram(gpuHistogram);
  printPercentiles(gpuHistogram);

  std::printf("%s\n", allCorrect ? "ALL PASS" : "SOME TESTS FAILED");
  return allCorrect ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_trip_durations.cu -o bikeshare_trip_durations && ./bikeshare_trip_durations

## `histogram.cu`

In [ ]:
%%writefile histogram.cu
#include "cuda_compat.h"

#include <cstdint>
#include <cstdio>
#include <vector>

constexpr int ALPHABET_SIZE = 26;
constexpr int LETTERS_PER_BIN = 4;
constexpr unsigned NUM_BINS = (ALPHABET_SIZE + LETTERS_PER_BIN - 1) / LETTERS_PER_BIN;

constexpr int NOT_A_LETTER = -1;
constexpr unsigned THREADS_PER_BLOCK = 256;
constexpr unsigned CFACTOR = 32;

#ifndef CPU_CHARS
#define CPU_CHARS (1u << 18)
#endif
const unsigned TEXT_LENGTH = BY_BACKEND(1u << 26, CPU_CHARS);
const int TIMED_RUNS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

__host__ __device__ inline int binOf(char c) {
  int alphabet_position = c - 'a';
  if (alphabet_position >= 0 && alphabet_position < ALPHABET_SIZE) {
    return alphabet_position / LETTERS_PER_BIN;
  }
  return NOT_A_LETTER;
}

__global__ void histo_basic(const char* data, unsigned length, unsigned* histo) {
  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < length) {
    int b = binOf(data[i]);
    if (b >= 0) {
      atomicAdd(&histo[b], 1u);
    }
  }
}

__global__ void histo_private_global(const char* data, unsigned length, unsigned* histo) {
  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < length) {
    int b = binOf(data[i]);
    if (b >= 0) {
      atomicAdd(&histo[blockIdx.x * NUM_BINS + b], 1u);
    }
  }
  __syncthreads();

  if (blockIdx.x > 0) {
    for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
      unsigned v = histo[blockIdx.x * NUM_BINS + bin];
      if (v > 0) {
        atomicAdd(&histo[bin], v);
      }
    }
  }
}

__global__ void histo_private_shared(const char* data, unsigned length, unsigned* histo) {
  __shared__ unsigned histo_s[NUM_BINS];
  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    histo_s[bin] = 0u;
  }
  __syncthreads();

  unsigned i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < length) {
    int b = binOf(data[i]);
    if (b >= 0) {
      atomicAdd(&histo_s[b], 1u);
    }
  }
  __syncthreads();

  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    unsigned v = histo_s[bin];
    if (v > 0) {
      atomicAdd(&histo[bin], v);
    }
  }
}

__global__ void histo_coarse_contig(const char* data, unsigned length, unsigned* histo) {
  __shared__ unsigned histo_s[NUM_BINS];
  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    histo_s[bin] = 0u;
  }
  __syncthreads();

  unsigned tid = blockIdx.x * blockDim.x + threadIdx.x;
  unsigned begin = tid * CFACTOR;
  unsigned end = begin + CFACTOR;
  if (end > length) {
    end = length;
  }
  for (unsigned i = begin; i < end; ++i) {
    int b = binOf(data[i]);
    if (b >= 0) {
      atomicAdd(&histo_s[b], 1u);
    }
  }
  __syncthreads();

  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    unsigned v = histo_s[bin];
    if (v > 0) {
      atomicAdd(&histo[bin], v);
    }
  }
}

__global__ void histo_coarse_interl(const char* data, unsigned length, unsigned* histo) {
  __shared__ unsigned histo_s[NUM_BINS];
  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    histo_s[bin] = 0u;
  }
  __syncthreads();

  unsigned tid = blockIdx.x * blockDim.x + threadIdx.x;
  for (unsigned i = tid; i < length; i += blockDim.x * gridDim.x) {
    int b = binOf(data[i]);
    if (b >= 0) {
      atomicAdd(&histo_s[b], 1u);
    }
  }
  __syncthreads();

  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    unsigned v = histo_s[bin];
    if (v > 0) {
      atomicAdd(&histo[bin], v);
    }
  }
}

__global__ void histo_aggregation(const char* data, unsigned length, unsigned* histo) {
  __shared__ unsigned histo_s[NUM_BINS];
  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    histo_s[bin] = 0u;
  }
  __syncthreads();

  unsigned accumulator = 0;
  int prevBin = -1;
  unsigned tid = blockIdx.x * blockDim.x + threadIdx.x;
  for (unsigned i = tid; i < length; i += blockDim.x * gridDim.x) {
    int b = binOf(data[i]);
    if (b >= 0) {
      if (b == prevBin) {
        ++accumulator;
      } else {
        if (accumulator > 0) {
          atomicAdd(&histo_s[prevBin], accumulator);
        }
        accumulator = 1;
        prevBin = b;
      }
    }
  }
  if (accumulator > 0) {
    atomicAdd(&histo_s[prevBin], accumulator);
  }
  __syncthreads();

  for (unsigned bin = threadIdx.x; bin < NUM_BINS; bin += blockDim.x) {
    unsigned v = histo_s[bin];
    if (v > 0) {
      atomicAdd(&histo[bin], v);
    }
  }
}

static void histogram_sequential(const char* data, unsigned length, unsigned* histo) {
  for (unsigned b = 0; b < NUM_BINS; ++b) {
    histo[b] = 0;
  }

  for (unsigned i = 0; i < length; ++i) {
    int alphabet_position = data[i] - 'a';
    if (alphabet_position >= 0 && alphabet_position < ALPHABET_SIZE) {
      histo[alphabet_position / LETTERS_PER_BIN]++;
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  unsigned below(unsigned bound) {
    return static_cast<unsigned>(next() % bound);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

constexpr unsigned ENGLISH_LETTER_FREQUENCY_PER_10000[ALPHABET_SIZE] = {
    817, 149, 278, 425, 1270, 223, 202, 609, 697, 15,  77, 403, 241,
    675, 751, 193, 10,  599,  633, 906, 276, 98,  236, 15, 197, 7,
};

constexpr unsigned sumOfLetterFrequencies() {
  unsigned total = 0;
  for (unsigned frequency : ENGLISH_LETTER_FREQUENCY_PER_10000) {
    total += frequency;
  }
  return total;
}

constexpr unsigned LETTER_FREQUENCY_TOTAL = sumOfLetterFrequencies();
static_assert(LETTER_FREQUENCY_TOTAL == 10002, "the rounded frequencies add up to 10002");

constexpr char MOST_FREQUENT_LETTER = 'e';
constexpr unsigned MAX_WORD_LENGTH = 10;
constexpr unsigned ONE_SENTENCE_END_EVERY_N_WORDS = 12;
constexpr char SKY_LETTER = 'a';
constexpr unsigned SKY_PERCENT = 95;
constexpr unsigned ONE_HUNDRED_PERCENT = 100;
constexpr uint64_t ENGLISH_TEXT_SEED = 2024;
constexpr uint64_t SKY_TEXT_SEED = 7;

static char randomEnglishLetter(SplitMix64& rng) {
  unsigned draw = rng.below(LETTER_FREQUENCY_TOTAL);
  unsigned cumulativeFrequency = 0;
  for (int letter = 0; letter < ALPHABET_SIZE; ++letter) {
    cumulativeFrequency += ENGLISH_LETTER_FREQUENCY_PER_10000[letter];
    if (draw < cumulativeFrequency) {
      return static_cast<char>('a' + letter);
    }
  }
  return MOST_FREQUENT_LETTER;
}

static std::vector<char> makeEnglishLikeText(unsigned length, uint64_t seed) {
  SplitMix64 rng(seed);
  std::vector<char> text;
  text.reserve(length);
  auto hasRoom = [&] {
    return text.size() < length;
  };

  while (hasRoom()) {
    unsigned wordLength = 1 + rng.below(MAX_WORD_LENGTH);
    for (unsigned k = 0; k < wordLength && hasRoom(); ++k) {
      text.push_back(randomEnglishLetter(rng));
    }
    if (hasRoom() && rng.below(ONE_SENTENCE_END_EVERY_N_WORDS) == 0) {
      text.push_back('.');
    }
    if (hasRoom()) {
      text.push_back(' ');
    }
  }
  return text;
}

static std::vector<char> makeSkyText(unsigned length, uint64_t seed) {
  SplitMix64 rng(seed);
  std::vector<char> text(length);
  for (char& c : text) {
    bool isSky = rng.below(ONE_HUNDRED_PERCENT) < SKY_PERCENT;
    c = isSky ? SKY_LETTER : randomEnglishLetter(rng);
  }
  return text;
}

using Histogram = std::vector<unsigned>;

static Histogram sequentialReference(const std::vector<char>& text, double& elapsedMs) {
  Histogram reference(NUM_BINS);
  Timer timer;
  histogram_sequential(text.data(), static_cast<unsigned>(text.size()), reference.data());
  elapsedMs = timer.ms();
  return reference;
}

static unsigned long long totalCount(const Histogram& histo) {
  unsigned long long total = 0;
  for (unsigned count : histo) {
    total += count;
  }
  return total;
}

enum class FirstCopyIsFinal { No, Yes };

template <class BlockOfCharacter>
static unsigned long long countCommitAtomics(const std::vector<char>& text,
                                             unsigned numBlocks,
                                             BlockOfCharacter blockOfCharacter,
                                             FirstCopyIsFinal firstCopyIsFinal) {
  std::vector<bool> binIsUsed(static_cast<size_t>(numBlocks) * NUM_BINS, false);
  for (unsigned i = 0; i < text.size(); ++i) {
    int b = binOf(text[i]);
    if (b >= 0) {
      binIsUsed[static_cast<size_t>(blockOfCharacter(i)) * NUM_BINS + b] = true;
    }
  }

  size_t firstCommittedCounter = firstCopyIsFinal == FirstCopyIsFinal::Yes ? NUM_BINS : 0;
  unsigned long long commits = 0;
  for (size_t counter = firstCommittedCounter; counter < binIsUsed.size(); ++counter) {
    if (binIsUsed[counter]) {
      ++commits;
    }
  }
  return commits;
}

static unsigned long long countAggregatedAtomics(const std::vector<char>& text,
                                                 unsigned threadCount) {
  std::vector<int> previousBinOfThread(threadCount, NOT_A_LETTER);
  unsigned long long runs = 0;
  for (unsigned i = 0; i < text.size(); ++i) {
    int b = binOf(text[i]);
    int& previousBin = previousBinOfThread[i % threadCount];
    if (b >= 0 && b != previousBin) {
      ++runs;
      previousBin = b;
    }
  }
  return runs;
}

constexpr int BAR_WIDTH = 50;
const char* const BIN_LABELS[NUM_BINS] = {"a-d", "e-h", "i-l", "m-p", "q-t", "u-x", "y-z"};

static void printBar(unsigned count, unsigned long long total) {
  int width = total ? static_cast<int>(BAR_WIDTH * static_cast<double>(count) / total + 0.5) : 0;
  for (int j = 0; j < width; ++j) {
    std::putchar('#');
  }
  std::putchar('\n');
}

static void printLetterHistogram(const Histogram& histo) {
  const unsigned long long total = totalCount(histo);
  for (unsigned b = 0; b < NUM_BINS; ++b) {
    std::printf("  %s %10u  ", BIN_LABELS[b], histo[b]);
    printBar(histo[b], total);
  }
}

using HistoKernel = void (*)(const char*, unsigned, unsigned*);

struct KernelVariant {
  const char* name;
  HistoKernel kernel;
  unsigned blocks;
  size_t countersToClear;
  unsigned long long globalAtomics;
  unsigned long long sharedAtomics;
};

struct LaunchShape {
  unsigned fineBlocks;
  unsigned coarseBlocks;
  unsigned coarseThreads;
};

static LaunchShape launchShapeFor(unsigned length) {
  LaunchShape shape;
  shape.fineBlocks = cdiv(length, THREADS_PER_BLOCK);
  shape.coarseBlocks = cdiv(length, THREADS_PER_BLOCK * CFACTOR);
  shape.coarseThreads = shape.coarseBlocks * THREADS_PER_BLOCK;
  return shape;
}

static std::vector<KernelVariant> kernelVariants(const std::vector<char>& text,
                                                 const LaunchShape& shape,
                                                 unsigned long long letters) {
  const unsigned coarseThreads = shape.coarseThreads;
  auto blockOfOneCharacterPerThread = [](unsigned i) {
    return i / THREADS_PER_BLOCK;
  };
  auto blockOfContiguousChunk = [](unsigned i) {
    return (i / CFACTOR) / THREADS_PER_BLOCK;
  };
  auto blockOfInterleavedCharacter = [=](unsigned i) {
    return (i % coarseThreads) / THREADS_PER_BLOCK;
  };

  const unsigned long long fineCommits = countCommitAtomics(
      text, shape.fineBlocks, blockOfOneCharacterPerThread, FirstCopyIsFinal::No);
  const unsigned long long fineCommitsIntoFirstCopy = countCommitAtomics(
      text, shape.fineBlocks, blockOfOneCharacterPerThread, FirstCopyIsFinal::Yes);
  const unsigned long long contiguousCommits =
      countCommitAtomics(text, shape.coarseBlocks, blockOfContiguousChunk, FirstCopyIsFinal::No);
  const unsigned long long interleavedCommits = countCommitAtomics(
      text, shape.coarseBlocks, blockOfInterleavedCharacter, FirstCopyIsFinal::No);
  const unsigned long long aggregatedAtomics = countAggregatedAtomics(text, coarseThreads);
  const size_t oneCopyPerBlock = static_cast<size_t>(shape.fineBlocks) * NUM_BINS;

  return {
      {"1 basic (global atomics)", histo_basic, shape.fineBlocks, NUM_BINS, letters, 0},
      {"2 private copies, global",
       histo_private_global,
       shape.fineBlocks,
       oneCopyPerBlock,
       letters + fineCommitsIntoFirstCopy,
       0},
      {"3 private copy, shared",
       histo_private_shared,
       shape.fineBlocks,
       NUM_BINS,
       fineCommits,
       letters},
      {"4 + coarsening contiguous",
       histo_coarse_contig,
       shape.coarseBlocks,
       NUM_BINS,
       contiguousCommits,
       letters},
      {"5 + coarsening interleaved",
       histo_coarse_interl,
       shape.coarseBlocks,
       NUM_BINS,
       interleavedCommits,
       letters},
      {"6 + aggregation",
       histo_aggregation,
       shape.coarseBlocks,
       NUM_BINS,
       interleavedCommits,
       aggregatedAtomics},
  };
}

struct DeviceBuffers {
  char* text;
  unsigned* histo;
  unsigned length;
};

static DeviceBuffers copyTextToDevice(const std::vector<char>& text, size_t histoCounters) {
  DeviceBuffers device;
  device.length = static_cast<unsigned>(text.size());

  CHECK(cudaMalloc(&device.text, device.length));
  CHECK(cudaMemcpy(device.text, text.data(), device.length, cudaMemcpyHostToDevice));

  CHECK(cudaMalloc(&device.histo, histoCounters * sizeof(unsigned)));
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.text));
  CHECK(cudaFree(device.histo));
}

static double timeKernel(const KernelVariant& variant, const DeviceBuffers& device) {
  auto clearAndLaunch = [&] {
    CHECK(cudaMemset(device.histo, 0, variant.countersToClear * sizeof(unsigned)));
    launch(variant.kernel,
           dim3(variant.blocks),
           dim3(THREADS_PER_BLOCK),
           device.text,
           device.length,
           device.histo);
  };

  if (WARM_UP_BEFORE_TIMING) {
    clearAndLaunch();
  }
  CHECK(cudaGetLastError());

  Timer timer;
  for (int run = 0; run < TIMED_RUNS; ++run) {
    clearAndLaunch();
  }
  double ms = timer.ms() / TIMED_RUNS;
  CHECK(cudaGetLastError());
  return ms;
}

static Histogram copyHistogramToHost(const DeviceBuffers& device) {
  Histogram histo(NUM_BINS);
  CHECK(
      cudaMemcpy(histo.data(), device.histo, NUM_BINS * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return histo;
}

static void printTableHeader(double cpuMs) {
  std::printf("%-28s %8s %9s %15s %15s  %s\n",
              "kernel",
              "blocks",
              "ms",
              "global atomics",
              "shared atomics",
              "check");
  std::printf(
      "%-28s %8s %9.3f %15s %15s  %s\n", "CPU sequential", "-", cpuMs, "-", "-", "reference");
}

static void printVariantRow(const KernelVariant& variant, double ms, bool correct) {
  std::printf("%-28s %8u %9.3f %15llu %15llu  %s\n",
              variant.name,
              variant.blocks,
              ms,
              variant.globalAtomics,
              variant.sharedAtomics,
              correct ? "PASS" : "FAIL");
}

static bool runInput(const char* label, const std::vector<char>& text) {
  double cpuMs;
  const Histogram reference = sequentialReference(text, cpuMs);
  const unsigned long long letters = totalCount(reference);

  std::printf("\n=== input: %s, %zu characters, %llu letters ===\n", label, text.size(), letters);
  printLetterHistogram(reference);

  const LaunchShape shape = launchShapeFor(static_cast<unsigned>(text.size()));
  const std::vector<KernelVariant> variants = kernelVariants(text, shape, letters);
  const DeviceBuffers device =
      copyTextToDevice(text, static_cast<size_t>(shape.fineBlocks) * NUM_BINS);

  printTableHeader(cpuMs);
  bool allCorrect = true;
  for (const KernelVariant& variant : variants) {
    const double ms = timeKernel(variant, device);
    const bool correct = copyHistogramToHost(device) == reference;
    allCorrect = allCorrect && correct;
    printVariantRow(variant, ms, correct);
  }

  freeDeviceBuffers(device);
  return allCorrect;
}

int main() {
  std::printf("Parallel histogram (chapter 9), backend: %s\n", CUDA_BACKEND);
  std::printf(
      "%u bins of %d letters, %u threads per block, coarsening factor %u, %d timed run(s)\n",
      NUM_BINS,
      LETTERS_PER_BIN,
      THREADS_PER_BLOCK,
      CFACTOR,
      TIMED_RUNS);

  bool allPass = runInput("English-like text", makeEnglishLikeText(TEXT_LENGTH, ENGLISH_TEXT_SEED));
  allPass = runInput("sky (95% in one bin)", makeSkyText(TEXT_LENGTH, SKY_TEXT_SEED)) && allPass;

  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME FAILED");
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. histogram.cu -o histogram && ./histogram

## `histogram_image.cu`

In [ ]:
%%writefile histogram_image.cu
#include "cuda_compat.h"

#include <cstdint>
#include <cstdio>
#include <vector>

constexpr unsigned NUM_INTENSITIES = 256;
constexpr unsigned NUM_BINS = NUM_INTENSITIES;
constexpr unsigned NUM_BINS_SMALL = 16;
constexpr unsigned INTENSITIES_PER_SMALL_BIN = NUM_INTENSITIES / NUM_BINS_SMALL;
constexpr unsigned COARSE_FACTOR = 32;
constexpr unsigned THREADS_PER_BLOCK = 256;

const unsigned WIDTH = BY_BACKEND(4096u, 512u);
const unsigned HEIGHT = BY_BACKEND(4096u, 512u);
const int TIMED_RUNS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);
const bool RUNNING_ON_CPU_EMULATION = BY_BACKEND(false, true);

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i];
    ++bins[b];
  }
}

__global__ void histogram_basic(const unsigned char* image,
                                unsigned int* bins,
                                unsigned int width,
                                unsigned int height) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i];
    cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
    bins_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
}

__global__ void histogram_private_global(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int* bins_pool,
                                         unsigned int width,
                                         unsigned int height) {
  unsigned int* bins_priv = &bins_pool[blockIdx.x * NUM_BINS];

  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i];
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_priv_ref(bins_priv[b]);
    bins_priv_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_priv[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_priv[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_private_shared(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int width,
                                         unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i];
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
    bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_coarse_contig(const unsigned char* image,
                                        unsigned int* bins,
                                        unsigned int width,
                                        unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + threadIdx.x * COARSE_FACTOR + c;
    if (i < width * height) {
      unsigned char b = image[i];
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_coarse_interl(const unsigned char* image,
                                        unsigned int* bins,
                                        unsigned int width,
                                        unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + c * blockDim.x + threadIdx.x;
    if (i < width * height) {
      unsigned char b = image[i];
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_thread_private(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int width,
                                         unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS_SMALL];
  for (unsigned int b = threadIdx.x; b < NUM_BINS_SMALL; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int bins_thread[NUM_BINS_SMALL];
  for (unsigned int b = 0; b < NUM_BINS_SMALL; ++b) {
    bins_thread[b] = 0;
  }

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + c * blockDim.x + threadIdx.x;
    if (i < width * height) {
      ++bins_thread[image[i] / INTENSITIES_PER_SMALL_BIN];
    }
  }

  for (unsigned int b = 0; b < NUM_BINS_SMALL; ++b) {
    if (bins_thread[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(bins_thread[b], cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS_SMALL; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_thread_run(const unsigned char* image,
                                     unsigned int* bins,
                                     unsigned int width,
                                     unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  if (segment + threadIdx.x < width * height) {
    unsigned int b = image[segment + threadIdx.x];
    unsigned int bin_r = 1;
    for (unsigned int c = 1; c < COARSE_FACTOR; ++c) {
      unsigned int i = segment + c * blockDim.x + threadIdx.x;
      if (i < width * height) {
        unsigned char bNext = image[i];
        if (bNext == b) {
          ++bin_r;
        } else {
          cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
          bins_s_ref.fetch_add(bin_r, cuda::memory_order_relaxed);
          b = bNext;
          bin_r = 1;
        }
      }
    }
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
    bins_s_ref.fetch_add(bin_r, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  unsigned below(unsigned bound) {
    return static_cast<unsigned>(next() % bound);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

using Image = std::vector<unsigned char>;
using Histogram = std::vector<unsigned>;

constexpr uint64_t NOISE_SEED = 11;
constexpr uint64_t PHOTO_SEED = 7;

static Image makeNoise(unsigned width, unsigned height) {
  SplitMix64 rng(NOISE_SEED);
  Image image(static_cast<size_t>(width) * height);
  for (unsigned char& pixel : image) {
    pixel = static_cast<unsigned char>(rng.below(NUM_INTENSITIES));
  }
  return image;
}

enum class Region { Grass, Leaves, Trunk, Sky };

struct IntensityRange {
  unsigned lowest;
  unsigned count;
};

constexpr IntensityRange GRASS_INTENSITIES = {150, 30};
constexpr IntensityRange LEAVES_INTENSITIES = {90, 30};
constexpr IntensityRange TRUNK_INTENSITIES = {0, 40};
constexpr IntensityRange SKY_GLARE_INTENSITIES = {225, 31};
constexpr unsigned char PLAIN_SKY_INTENSITY = 240;
constexpr unsigned ONE_GLARE_PIXEL_EVERY = 10;

constexpr unsigned FIFTHS_OF_HEIGHT = 5;
constexpr unsigned GRASS_TOP_FIFTHS_DOWN = 4;
constexpr unsigned CROWN_CENTER_FIFTHS_DOWN = 2;
constexpr unsigned CROWN_RADIUS_DIVISOR = 5;
constexpr unsigned TRUNK_HALF_WIDTH_DIVISOR = 40;

static unsigned fifthsDown(unsigned height, unsigned fifths) {
  return height * fifths / FIFTHS_OF_HEIGHT;
}

static Region regionOf(unsigned x, unsigned y, unsigned width, unsigned height) {
  const unsigned crownCenterX = width / 2;
  const unsigned crownCenterY = fifthsDown(height, CROWN_CENTER_FIFTHS_DOWN);
  const int dx = static_cast<int>(x) - static_cast<int>(crownCenterX);
  const int dy = static_cast<int>(y) - static_cast<int>(crownCenterY);
  const int crownRadiusSquared =
      static_cast<int>(width * width / (CROWN_RADIUS_DIVISOR * CROWN_RADIUS_DIVISOR));
  const int trunkHalfWidth = static_cast<int>(width / TRUNK_HALF_WIDTH_DIVISOR);

  if (y > fifthsDown(height, GRASS_TOP_FIFTHS_DOWN)) {
    return Region::Grass;
  }
  if (dx * dx + dy * dy < crownRadiusSquared) {
    return Region::Leaves;
  }
  const bool belowCrownCenter = y > crownCenterY;
  const int distanceFromTrunkAxis = dx < 0 ? -dx : dx;
  if (belowCrownCenter && distanceFromTrunkAxis < trunkHalfWidth) {
    return Region::Trunk;
  }
  return Region::Sky;
}

static unsigned char randomIntensity(IntensityRange range, SplitMix64& rng) {
  return static_cast<unsigned char>(range.lowest + rng.below(range.count));
}

static unsigned char intensityOf(Region region, SplitMix64& rng) {
  switch (region) {
    case Region::Grass:
      return randomIntensity(GRASS_INTENSITIES, rng);
    case Region::Leaves:
      return randomIntensity(LEAVES_INTENSITIES, rng);
    case Region::Trunk:
      return randomIntensity(TRUNK_INTENSITIES, rng);
    case Region::Sky:
      break;
  }
  const bool isGlare = rng.below(ONE_GLARE_PIXEL_EVERY) == 0;
  return isGlare ? randomIntensity(SKY_GLARE_INTENSITIES, rng) : PLAIN_SKY_INTENSITY;
}

static Image makePhoto(unsigned width, unsigned height) {
  SplitMix64 rng(PHOTO_SEED);
  Image image(static_cast<size_t>(width) * height);
  for (unsigned y = 0; y < height; ++y) {
    for (unsigned x = 0; x < width; ++x) {
      image[static_cast<size_t>(y) * width + x] = intensityOf(regionOf(x, y, width, height), rng);
    }
  }
  return image;
}

constexpr unsigned NUM_QUARTERS = 4;
constexpr unsigned BINS_PER_QUARTER = NUM_BINS / NUM_QUARTERS;
constexpr int BAR_WIDTH = 50;
const char* const QUARTER_LABELS[NUM_QUARTERS] = {
    "[0-63]   ",
    "[64-127] ",
    "[128-191]",
    "[192-255]",
};

static void printBar(unsigned long long count, unsigned long long total) {
  const int width = static_cast<int>(BAR_WIDTH * static_cast<double>(count) / total + 0.5);
  for (int j = 0; j < width; ++j) {
    std::putchar('#');
  }
  std::putchar('\n');
}

static void printQuarters(const Histogram& reference) {
  unsigned long long quarterCounts[NUM_QUARTERS] = {0, 0, 0, 0};
  unsigned long long total = 0;
  for (unsigned b = 0; b < NUM_BINS; ++b) {
    quarterCounts[b / BINS_PER_QUARTER] += reference[b];
    total += reference[b];
  }

  for (unsigned quarter = 0; quarter < NUM_QUARTERS; ++quarter) {
    std::printf("  %s %10llu  ", QUARTER_LABELS[quarter], quarterCounts[quarter]);
    printBar(quarterCounts[quarter], total);
  }
}

static unsigned fullestBin(const Histogram& histo) {
  unsigned fullest = 0;
  for (unsigned b = 1; b < histo.size(); ++b) {
    if (histo[b] > histo[fullest]) {
      fullest = b;
    }
  }
  return fullest;
}

static Histogram mergeIntoSmallBins(const Histogram& histo) {
  Histogram small(NUM_BINS_SMALL, 0);
  for (unsigned b = 0; b < NUM_BINS; ++b) {
    small[b / INTENSITIES_PER_SMALL_BIN] += histo[b];
  }
  return small;
}

enum class ImageKernel {
  Basic,
  PrivateGlobal,
  PrivateShared,
  CoarseContiguous,
  CoarseInterleaved,
  ThreadPrivate,
  ThreadRun,
};

enum class Coarsening { OnePixelPerThread, CoarseFactorPixelsPerThread };

struct KernelVariant {
  const char* name;
  ImageKernel kernel;
  Coarsening coarsening;
};

const KernelVariant VARIANTS[] = {
    {"1 basic (device atomics)", ImageKernel::Basic, Coarsening::OnePixelPerThread},
    {"2 private copies, global", ImageKernel::PrivateGlobal, Coarsening::OnePixelPerThread},
    {"3 private copy, shared", ImageKernel::PrivateShared, Coarsening::OnePixelPerThread},
    {"4 + coarsening contiguous",
     ImageKernel::CoarseContiguous,
     Coarsening::CoarseFactorPixelsPerThread},
    {"5 + coarsening interleaved",
     ImageKernel::CoarseInterleaved,
     Coarsening::CoarseFactorPixelsPerThread},
    {"6 thread-private, 16 bins",
     ImageKernel::ThreadPrivate,
     Coarsening::CoarseFactorPixelsPerThread},
    {"7 most recent bin private", ImageKernel::ThreadRun, Coarsening::CoarseFactorPixelsPerThread},
};

static unsigned numPixels() {
  return WIDTH * HEIGHT;
}

static unsigned blocksFor(Coarsening coarsening) {
  if (coarsening == Coarsening::OnePixelPerThread) {
    return cdiv(numPixels(), THREADS_PER_BLOCK);
  }
  return cdiv(numPixels(), THREADS_PER_BLOCK * COARSE_FACTOR);
}

struct DeviceBuffers {
  unsigned char* image;
  unsigned* bins;
  unsigned* bins_pool;
};

static size_t poolBytes(unsigned blocks) {
  return static_cast<size_t>(blocks) * NUM_BINS * sizeof(unsigned);
}

static DeviceBuffers copyImageToDevice(const Image& image) {
  DeviceBuffers device;
  CHECK(cudaMalloc(&device.image, numPixels()));
  CHECK(cudaMalloc(&device.bins, NUM_BINS * sizeof(unsigned)));
  CHECK(cudaMalloc(&device.bins_pool, poolBytes(blocksFor(Coarsening::OnePixelPerThread))));
  CHECK(cudaMemcpy(device.image, image.data(), numPixels(), cudaMemcpyHostToDevice));
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.image));
  CHECK(cudaFree(device.bins));
  CHECK(cudaFree(device.bins_pool));
}

static void clearAndLaunch(ImageKernel kernel, unsigned blocks, const DeviceBuffers& device) {
  const unsigned char* image = device.image;
  const dim3 grid(blocks);
  const dim3 block(THREADS_PER_BLOCK);

  CHECK(cudaMemset(device.bins, 0, NUM_BINS * sizeof(unsigned)));
  switch (kernel) {
    case ImageKernel::Basic:
      launch(histogram_basic, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
    case ImageKernel::PrivateGlobal:
      CHECK(cudaMemset(device.bins_pool, 0, poolBytes(blocks)));
      launch(histogram_private_global,
             grid,
             block,
             image,
             device.bins,
             device.bins_pool,
             WIDTH,
             HEIGHT);
      break;
    case ImageKernel::PrivateShared:
      launch(histogram_private_shared, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
    case ImageKernel::CoarseContiguous:
      launch(histogram_coarse_contig, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
    case ImageKernel::CoarseInterleaved:
      launch(histogram_coarse_interl, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
    case ImageKernel::ThreadPrivate:
      launch(histogram_thread_private, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
    case ImageKernel::ThreadRun:
      launch(histogram_thread_run, grid, block, image, device.bins, WIDTH, HEIGHT);
      break;
  }
}

static double timeKernel(ImageKernel kernel, unsigned blocks, const DeviceBuffers& device) {
  if (WARM_UP_BEFORE_TIMING) {
    clearAndLaunch(kernel, blocks, device);
  }
  CHECK(cudaGetLastError());

  Timer timer;
  for (int run = 0; run < TIMED_RUNS; ++run) {
    clearAndLaunch(kernel, blocks, device);
  }
  const double ms = timer.ms() / TIMED_RUNS;
  CHECK(cudaGetLastError());
  return ms;
}

static Histogram copyBinsToHost(const DeviceBuffers& device) {
  Histogram bins(NUM_BINS);
  CHECK(cudaMemcpy(bins.data(), device.bins, NUM_BINS * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return bins;
}

static bool firstBinsMatch(const Histogram& got, const Histogram& expected) {
  for (unsigned b = 0; b < expected.size(); ++b) {
    if (got[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

struct References {
  Histogram bins;
  Histogram smallBins;
  double cpuMs;
};

static References sequentialReferences(const Image& image) {
  References references;
  references.bins.assign(NUM_BINS, 0);
  Timer timer;
  histogram_sequential(image.data(), references.bins.data(), WIDTH, HEIGHT);
  references.cpuMs = timer.ms();
  references.smallBins = mergeIntoSmallBins(references.bins);
  return references;
}

static bool isCorrect(ImageKernel kernel, const Histogram& got, const References& references) {
  if (kernel == ImageKernel::ThreadPrivate) {
    return firstBinsMatch(got, references.smallBins);
  }
  return got == references.bins;
}

static void printImageSummary(const char* label, const Histogram& reference) {
  const unsigned fullest = fullestBin(reference);
  std::printf("\n=== %s: %ux%u pixels; fullest bin %u holds %.1f%% of the pixels ===\n",
              label,
              WIDTH,
              HEIGHT,
              fullest,
              100.0 * reference[fullest] / numPixels());
  printQuarters(reference);
}

static void printTableHeader(double cpuMs) {
  std::printf("%-28s %7s %9s  %s\n", "kernel", "blocks", "ms", "check");
  std::printf("%-28s %7s %9.3f  %s\n", "0 CPU sequential", "-", cpuMs, "reference");
}

static bool runImage(const char* label, const Image& image) {
  const References references = sequentialReferences(image);
  printImageSummary(label, references.bins);

  const DeviceBuffers device = copyImageToDevice(image);
  printTableHeader(references.cpuMs);

  bool allCorrect = true;
  for (const KernelVariant& variant : VARIANTS) {
    const unsigned blocks = blocksFor(variant.coarsening);
    const double ms = timeKernel(variant.kernel, blocks, device);
    const bool correct = isCorrect(variant.kernel, copyBinsToHost(device), references);
    allCorrect = allCorrect && correct;
    std::printf("%-28s %7u %9.3f  %s\n", variant.name, blocks, ms, correct ? "PASS" : "FAIL");
  }

  freeDeviceBuffers(device);
  return allCorrect;
}

int main() {
  std::printf("Image histogram with cuda::atomic_ref (chapter 9), backend: %s\n", CUDA_BACKEND);
  std::printf("%u bins, %u threads per block, coarsening factor %u, %d timed run(s)\n",
              NUM_BINS,
              THREADS_PER_BLOCK,
              COARSE_FACTOR,
              TIMED_RUNS);

  bool allPass = runImage("noise (uniform intensities)", makeNoise(WIDTH, HEIGHT));
  allPass = runImage("photo (tree under a bright sky)", makePhoto(WIDTH, HEIGHT)) && allPass;

  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME FAILED");
  if (RUNNING_ON_CPU_EMULATION) {
    std::printf("(CPU emulation: the times measure thread start-up and barriers, not the GPU.)\n");
  }
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. histogram_image.cu -o histogram_image && ./histogram_image

## `race_demo.cu`

In [ ]:
%%writefile race_demo.cu
#include "cuda_compat.h"

#include <cstdio>
#include <string>
#include <vector>

constexpr unsigned NUM_BINS = 256;
constexpr unsigned THREADS_PER_BLOCK = 256;
const unsigned NUM_PIXELS = BY_BACKEND(1u << 24, 1u << 16);

constexpr unsigned char TRUNK_INTENSITY = 20;
constexpr unsigned char LEAVES_INTENSITY = 100;
constexpr unsigned char GRASS_INTENSITY = 160;
constexpr unsigned char SKY_INTENSITY = 240;
constexpr unsigned NUM_INTENSITIES_IN_IMAGE = 4;
constexpr unsigned char IMAGE_INTENSITIES[NUM_INTENSITIES_IN_IMAGE] = {
    TRUNK_INTENSITY,
    LEAVES_INTENSITY,
    GRASS_INTENSITY,
    SKY_INTENSITY,
};

constexpr unsigned PIXEL_SCATTER_MULTIPLIER = 7;
constexpr unsigned PIXEL_SCATTER_DIVISOR = 3;

__device__ unsigned int readFromMemory(volatile unsigned int* bins, unsigned char b) {
  return bins[b];
}

__device__ void writeToMemory(volatile unsigned int* bins, unsigned char b, unsigned int value) {
  bins[b] = value;
}

__global__ void racy_kernel(const unsigned char* image, unsigned int* bins, unsigned int n) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    unsigned char b = image[i];

    unsigned int oldValue = readFromMemory(bins, b);
    unsigned int newValue = oldValue + 1;
    writeToMemory(bins, b, newValue);
  }
}

__global__ void atomic_kernel(const unsigned char* image, unsigned int* bins, unsigned int n) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    unsigned char b = image[i];
    cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
    bins_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
}

enum class KernelVersion { Racy, Atomic };

using Histogram = std::vector<unsigned>;

static unsigned char intensityOfPixel(unsigned pixel) {
  unsigned scattered = pixel * PIXEL_SCATTER_MULTIPLIER + pixel / PIXEL_SCATTER_DIVISOR;
  return IMAGE_INTENSITIES[scattered % NUM_INTENSITIES_IN_IMAGE];
}

static std::vector<unsigned char> makeImage() {
  std::vector<unsigned char> image(NUM_PIXELS);
  for (unsigned pixel = 0; pixel < NUM_PIXELS; ++pixel) {
    image[pixel] = intensityOfPixel(pixel);
  }
  return image;
}

static Histogram sequentialHistogram(const std::vector<unsigned char>& image) {
  Histogram bins(NUM_BINS, 0);
  for (unsigned char intensity : image) {
    ++bins[intensity];
  }
  return bins;
}

static void launchKernel(KernelVersion version, const unsigned char* d_image, unsigned* d_bins) {
  const dim3 grid(cdiv(NUM_PIXELS, THREADS_PER_BLOCK));
  const dim3 block(THREADS_PER_BLOCK);
  if (version == KernelVersion::Racy) {
    launch(racy_kernel, grid, block, d_image, d_bins, NUM_PIXELS);
  } else {
    launch(atomic_kernel, grid, block, d_image, d_bins, NUM_PIXELS);
  }
}

static Histogram runOnDevice(KernelVersion version,
                             const unsigned char* d_image,
                             unsigned* d_bins) {
  CHECK(cudaMemset(d_bins, 0, NUM_BINS * sizeof(unsigned)));
  launchKernel(version, d_image, d_bins);
  CHECK(cudaGetLastError());

  Histogram bins(NUM_BINS);
  CHECK(cudaMemcpy(bins.data(), d_bins, NUM_BINS * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return bins;
}

static unsigned long long lostUpdates(const Histogram& bins) {
  unsigned long long counted = 0;
  for (unsigned count : bins) {
    counted += count;
  }
  return static_cast<unsigned long long>(NUM_PIXELS) - counted;
}

static void printTableHeader() {
  std::printf("%-8s", "kernel");
  for (unsigned char intensity : IMAGE_INTENSITIES) {
    std::string column = "bin " + std::to_string(intensity);
    std::printf(" %12s", column.c_str());
  }
  std::printf(" %12s\n", "lost");
}

static void printBinsOfImage(const char* kernelName, const Histogram& bins) {
  std::printf("%-8s", kernelName);
  for (unsigned char intensity : IMAGE_INTENSITIES) {
    std::printf(" %12u", bins[intensity]);
  }
}

static void printReferenceRow(const Histogram& reference) {
  printBinsOfImage("correct", reference);
  std::printf(" %12s\n", "-");
}

static void printMeasuredRow(const char* kernelName, const Histogram& bins) {
  printBinsOfImage(kernelName, bins);
  std::printf(" %12llu\n", lostUpdates(bins));
}

int main() {
  std::printf("Read-modify-write race on a histogram (chapter 9), backend: %s\n", CUDA_BACKEND);

  const std::vector<unsigned char> image = makeImage();
  const Histogram reference = sequentialHistogram(image);

  unsigned char* d_image;
  unsigned* d_bins;
  CHECK(cudaMalloc(&d_image, NUM_PIXELS));
  CHECK(cudaMalloc(&d_bins, NUM_BINS * sizeof(unsigned)));
  CHECK(cudaMemcpy(d_image, image.data(), NUM_PIXELS, cudaMemcpyHostToDevice));

  std::printf("%u pixels, %u distinct values, %u threads per block\n\n",
              NUM_PIXELS,
              NUM_INTENSITIES_IN_IMAGE,
              THREADS_PER_BLOCK);
  printTableHeader();
  printReferenceRow(reference);

  const Histogram racyBins = runOnDevice(KernelVersion::Racy, d_image, d_bins);
  printMeasuredRow("racy", racyBins);

  const Histogram atomicBins = runOnDevice(KernelVersion::Atomic, d_image, d_bins);
  printMeasuredRow("atomic", atomicBins);

  CHECK(cudaFree(d_image));
  CHECK(cudaFree(d_bins));

  const bool atomicIsExact = atomicBins == reference;
  std::printf(
      "\nThe racy count is whatever the interleaving produced; the atomic one must be exact.\n");
  std::printf("%s\n", atomicIsExact ? "PASS (atomic kernel exact)" : "FAIL (atomic kernel wrong)");
  return atomicIsExact ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. race_demo.cu -o race_demo && ./race_demo